In [1]:
# !pip install h2o pandas requests matplotlib seaborn

import io, os, shutil, importlib.util, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import requests

import h2o
from h2o.automl import H2OAutoML

warnings.filterwarnings('ignore')
print('h2o version:', h2o.__version__)

h2o version: 3.46.0.10


In [2]:
RAW_CSV_URL = (
    'https://raw.githubusercontent.com/jason12102/MLOps-Final-Project'
    '/main/data_versioning/diabetes.csv'
)

WORK_DIR   = Path('data_versioning')
SCRIPT_DIR = WORK_DIR / 'scripts'
OUTPUT_DIR = Path('h2o_automl_results')
for d in [WORK_DIR, SCRIPT_DIR, OUTPUT_DIR]:
    d.mkdir(exist_ok=True)

TARGET           = 'Outcome'
MAX_MODELS       = 20
MAX_RUNTIME_SECS = 120    # raise to 300-600 for a real run
SEED             = 42

In [3]:
resp = requests.get(RAW_CSV_URL, timeout=30)
resp.raise_for_status()

raw_csv = WORK_DIR / 'diabetes.csv'
raw_csv.write_bytes(resp.content)   # write_bytes avoids encoding issues
print(f'Fetched {raw_csv}  ({len(resp.content):,} bytes)')
pd.read_csv(raw_csv).head(3)

Fetched data_versioning\diabetes.csv  (23,875 bytes)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


In [4]:
clean_src = (
    'from pathlib import Path\n'
    'import pandas as pd\n'
    'CSV = Path(__file__).resolve().parents[1] / "diabetes.csv"\n'
    'ZERO_AS_MISSING = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]\n'
    'def main():\n'
    '    df = pd.read_csv(CSV)\n'
    '    for col in ZERO_AS_MISSING:\n'
    '        median = df.loc[df[col] != 0, col].median()\n'
    '        df.loc[df[col] == 0, col] = median\n'
    '    df.to_csv(CSV, index=False)\n'
    '    print("clean.py done")\n'
    'if __name__ == "__main__":\n'
    '    main()\n'
)

features_src = (
    'from pathlib import Path\n'
    'import pandas as pd\n'
    'CSV = Path(__file__).resolve().parents[1] / "diabetes.csv"\n'
    'BMI_BINS   = [float("-inf"), 18.5, 25, 30, float("inf")]\n'
    'BMI_LABELS = ["Underweight","Normal","Overweight","Obese"]\n'
    'AGE_BINS   = [float("-inf"), 29, 39, 49, 59, float("inf")]\n'
    'AGE_LABELS = ["20s","30s","40s","50s","60+"]\n'
    'def main():\n'
    '    df = pd.read_csv(CSV)\n'
    '    df["BMI_category"] = pd.cut(df["BMI"], bins=BMI_BINS, labels=BMI_LABELS, right=False)\n'
    '    df["Age_group"]    = pd.cut(df["Age"],  bins=AGE_BINS,  labels=AGE_LABELS)\n'
    '    df["Glucose_BMI"]  = df["Glucose"] * df["BMI"]\n'
    '    df.to_csv(CSV, index=False)\n'
    '    print("features.py done")\n'
    'if __name__ == "__main__":\n'
    '    main()\n'
)

(SCRIPT_DIR / 'clean.py').write_text(clean_src,     encoding='utf-8')
(SCRIPT_DIR / 'features.py').write_text(features_src, encoding='utf-8')
print('Scripts written.')

Scripts written.


In [5]:
def run_script(path):
    spec = importlib.util.spec_from_file_location('_mod', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.main()

# v1: raw snapshot (before any mutation)
v1_path = WORK_DIR / 'diabetes_v1_raw.csv'
shutil.copy(raw_csv, v1_path)

# v2: after median imputation
run_script(SCRIPT_DIR / 'clean.py')
v2_path = WORK_DIR / 'diabetes_v2_cleaned.csv'
shutil.copy(raw_csv, v2_path)

# v3: after feature engineering
run_script(SCRIPT_DIR / 'features.py')
v3_path = WORK_DIR / 'diabetes_v3_features.csv'
shutil.copy(raw_csv, v3_path)

VERSIONS = {
    'v1_raw':      v1_path,
    'v2_cleaned':  v2_path,
    'v3_features': v3_path,
}
dfs = {k: pd.read_csv(p) for k, p in VERSIONS.items()}
print('Snapshots ready.')

clean.py done
features.py done
Snapshots ready.


In [6]:
IMPUTED_COLS = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
for ver, df in dfs.items():
    zeros   = {c: int((df[c]==0).sum()) for c in IMPUTED_COLS}
    new_cols = [c for c in df.columns if c not in dfs['v1_raw'].columns]
    print(f'{ver:15s} | shape={str(df.shape):12s} | zeros={zeros} | new_cols={new_cols}')

v1_raw          | shape=(768, 9)     | zeros={'Glucose': 5, 'BloodPressure': 35, 'SkinThickness': 227, 'Insulin': 374, 'BMI': 11} | new_cols=[]
v2_cleaned      | shape=(768, 9)     | zeros={'Glucose': 0, 'BloodPressure': 0, 'SkinThickness': 0, 'Insulin': 0, 'BMI': 0} | new_cols=[]
v3_features     | shape=(768, 12)    | zeros={'Glucose': 0, 'BloodPressure': 0, 'SkinThickness': 0, 'Insulin': 0, 'BMI': 0} | new_cols=['BMI_category', 'Age_group', 'Glucose_BMI']


In [7]:
n         = len(dfs['v1_raw'])
rng       = np.random.default_rng(SEED)
test_idx  = sorted(rng.choice(n, size=int(n * 0.15), replace=False).tolist())
train_idx = [i for i in range(n) if i not in set(test_idx)]

print(f'Total rows : {n}')
print(f'Train      : {len(train_idx)}')
print(f'Test (held): {len(test_idx)}')

Total rows : 768
Train      : 653
Test (held): 115


In [8]:
h2o.init(nthreads=-1, max_mem_size='4G')
h2o.no_progress()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 24+36-3646, mixed mode, sharing)
  Starting server from C:\Users\Joshua\anaconda3\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\Joshua\AppData\Local\Temp\tmpzap3rn_b
  JVM stdout: C:\Users\Joshua\AppData\Local\Temp\tmpzap3rn_b\h2o_Joshua_started_from_python.out
  JVM stderr: C:\Users\Joshua\AppData\Local\Temp\tmpzap3rn_b\h2o_Joshua_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,America/Chicago
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,2 months and 8 days
H2O_cluster_name:,H2O_from_python_Joshua_qp6fvy
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.984 Gb
H2O_cluster_total_cores:,22
H2O_cluster_allowed_cores:,22
H2O_cluster_status:,"locked, healthy"


In [9]:
# ── Helper: safely extract scalar metrics from H2O's nested-list returns ──
def best_val(pairs):
    """pairs is [[threshold, value], ...] — return the value at the best threshold."""
    return max(v for _, v in pairs)

def extract_metrics(perf, model, ver):
    # Accuracy: 1 - minimum mean_per_class_error across thresholds
    acc = 1 - min(v for _, v in perf.mean_per_class_error())
    return {
        'version':   ver,
        'model_id':  model.model_id,
        'AUC':       round(perf.auc(),        4),
        'AUCPR':     round(perf.aucpr(),       4),
        'Logloss':   round(perf.logloss(),     4),
        'F1':        round(best_val(perf.F1()),        4),
        'Accuracy':  round(acc,                4),
        'Precision': round(best_val(perf.precision()), 4),
        'Recall':    round(best_val(perf.recall()),    4),
    }

results      = {}
summary_rows = []

for ver, df in dfs.items():
    print(f'\n{"="*58}')
    print(f'  AutoML -> {ver.upper()}')
    print(f'{"="*58}')

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    hf_train = h2o.H2OFrame(df_train)
    hf_test  = h2o.H2OFrame(df_test)
    hf_train[TARGET] = hf_train[TARGET].asfactor()
    hf_test[TARGET]  = hf_test[TARGET].asfactor()

    for cat in ['BMI_category', 'Age_group']:
        if cat in hf_train.columns:
            hf_train[cat] = hf_train[cat].asfactor()
            hf_test[cat]  = hf_test[cat].asfactor()

    features = [c for c in hf_train.columns if c != TARGET]
    print(f'  train={hf_train.nrow}  test={hf_test.nrow}  features({len(features)}): {features}')

    aml = H2OAutoML(
        max_models=MAX_MODELS,
        max_runtime_secs=MAX_RUNTIME_SECS,
        seed=SEED,
        project_name=f'diabetes_{ver}',
        sort_metric='AUC',
        balance_classes=True,
    )
    aml.train(x=features, y=TARGET, training_frame=hf_train)

    perf    = aml.leader.model_performance(hf_test)
    metrics = extract_metrics(perf, aml.leader, ver)
    summary_rows.append(metrics)
    results[ver] = {'aml': aml, 'perf': perf, 'hf_test': hf_test}

    lb = aml.leaderboard.as_data_frame()
    lb.to_csv(OUTPUT_DIR / f'leaderboard_{ver}.csv', index=False)

    print(f'\n  Best model : {aml.leader.model_id}')
    for k in ['AUC','AUCPR','F1','Accuracy','Precision','Recall','Logloss']:
        print(f'    {k:<10}: {metrics[k]}')
    print(f'\n  Top 5 Leaderboard:')
    display(lb.head(5))


  AutoML -> V1_RAW
  train=653  test=115  features(8): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

23:57:16.269: AutoML: XGBoost is not available; skipping it.


  Best model : GBM_1_AutoML_1_20260520_235716
    AUC       : 0.7756
    AUCPR     : 0.6417
    F1        : 0.6471
    Accuracy  : 0.7495
    Precision : 1.0
    Recall    : 1.0
    Logloss   : 0.5196

  Top 5 Leaderboard:


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,GBM_1_AutoML_1_20260520_235716,0.843818,0.465955,0.741230,0.209769,0.392169,0.153797
1,GBM_5_AutoML_1_20260520_235716,0.838270,0.482276,0.725982,0.226363,0.398067,0.158457
2,GLM_1_AutoML_1_20260520_235716,0.837270,0.482346,0.716179,0.234579,0.397008,0.157616
3,GBM_2_AutoML_1_20260520_235716,0.834083,0.482920,0.718951,0.237603,0.400800,0.160641
4,DeepLearning_1_AutoML_1_20260520_235716,0.834027,0.482223,0.733106,0.241407,0.399909,0.159927



  AutoML -> V2_CLEANED
  train=653  test=115  features(8): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

23:59:24.728: AutoML: XGBoost is not available; skipping it.


  Best model : GBM_5_AutoML_2_20260520_235924
    AUC       : 0.7977
    AUCPR     : 0.6149
    F1        : 0.6304
    Accuracy  : 0.7475
    Precision : 0.8889
    Recall    : 1.0
    Logloss   : 0.4873

  Top 5 Leaderboard:


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,GBM_5_AutoML_2_20260520_235924,0.843222,0.472892,0.742462,0.220815,0.396080,0.156879
1,GBM_1_AutoML_2_20260520_235924,0.842650,0.469134,0.724705,0.207438,0.393405,0.154767
2,GBM_2_AutoML_2_20260520_235924,0.842033,0.472049,0.735067,0.227975,0.396011,0.156824
3,GLM_1_AutoML_2_20260520_235924,0.840932,0.476271,0.717475,0.235384,0.396091,0.156888
4,GBM_grid_1_AutoML_2_20260520_235924_model_2,0.840662,0.475035,0.733243,0.219953,0.396005,0.156820



  AutoML -> V3_FEATURES
  train=653  test=115  features(11): ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'BMI_category', 'Age_group', 'Glucose_BMI']

00:01:29.318: AutoML: XGBoost is not available; skipping it.


  Best model : GLM_1_AutoML_3_20260521_00129
    AUC       : 0.8286
    AUCPR     : 0.6575
    F1        : 0.6849
    Accuracy  : 0.7812
    Precision : 0.8889
    Recall    : 1.0
    Logloss   : 0.4691

  Top 5 Leaderboard:


,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,GLM_1_AutoML_3_20260521_00129,0.848801,0.465261,0.732319,0.220315,0.390597,0.152566
1,GBM_1_AutoML_3_20260521_00129,0.847072,0.460367,0.735891,0.207744,0.389764,0.151916
2,GBM_2_AutoML_3_20260521_00129,0.841564,0.474239,0.720567,0.224451,0.396539,0.157243
3,GBM_grid_1_AutoML_3_20260521_00129_model_4,0.840289,0.477262,0.706745,0.222314,0.398615,0.158894
4,DRF_1_AutoML_3_20260521_00129,0.837867,0.526943,0.722659,0.226338,0.399360,0.159488


In [13]:
for ver, res in results.items():
    path = h2o.save_model(
        model=res['aml'].leader,
        path=str(OUTPUT_DIR / f'best_model_{ver}'),
        force=True,
    )
    print(f'[{ver}] saved -> {path}')

[v1_raw] saved -> C:\Users\Joshua\Downloads\diabetes\h2o_automl_results\best_model_v1_raw\GBM_1_AutoML_1_20260520_235716
[v2_cleaned] saved -> C:\Users\Joshua\Downloads\diabetes\h2o_automl_results\best_model_v2_cleaned\GBM_5_AutoML_2_20260520_235924
[v3_features] saved -> C:\Users\Joshua\Downloads\diabetes\h2o_automl_results\best_model_v3_features\GLM_1_AutoML_3_20260521_00129
